In [1]:
from __future__ import annotations
import asyncio
from typing import Optional, List
from openapi_client.api.fleet_api import FleetApi
from openapi_client.api.agents_api import AgentsApi
from openapi_client.api.systems_api import SystemsApi
import math
import pandas as pd
from runtime_support import (
    setup_client_from_env,
    api_navigate_ship,
    api_get_ship_nav,
    api_purchase_cargo
    )
from core_helpers import (
    init_world_state
)

with setup_client_from_env() as client:
    fleet_api = FleetApi(client)
    agents_api = AgentsApi(client)
    systems_api = SystemsApi(client)

#Build all in-memory objects once (fleet_activity_obj, waypoints_ref_obj, waypoint_traits_obj, HQ)
state = await init_world_state(fleet_api, agents_api, systems_api)

In [ ]:
from runtime_support import api_dock_ship

await api_dock_ship(fleet_api, "KIJINIBIBI-1")

In [3]:
first_purchase = await api_purchase_cargo(fleet_api, "TROOTS-1", "FOOD", 1)
print(first_purchase)

cargo=ShipCargo(capacity=40, units=4, inventory=[ShipCargoItem(symbol=<TradeSymbol.IRON_ORE: 'IRON_ORE'>, name='Iron Ore', description='A common and valuable ore used in the production of steel and other alloys.', units=3), ShipCargoItem(symbol=<TradeSymbol.FOOD: 'FOOD'>, name='Galactic Cuisine', description='A diverse range of foods from different planets, including fresh produce, meats, and prepared meals.', units=1)]) transaction=MarketTransaction(waypoint_symbol='X1-Q51-K86', ship_symbol='TROOTS-1', trade_symbol='FOOD', type='PURCHASE', units=1, price_per_unit=1236, total_price=1236, timestamp=datetime.datetime(2025, 9, 14, 1, 0, 55, 945000, tzinfo=TzInfo(UTC))) agent=Agent(account_id='cmeb98xyw0028tm167bfnq149', symbol='TROOTS', headquarters='X1-Q51-A1', credits=136552, starting_faction='AEGIS', ship_count=2)


In [4]:
await api_navigate_ship(fleet_api, "TROOTS-1", "X1-Q51-A3")

starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Not at destination, continuing
Refuelling now...
Not in orbit, going into orbit now...
Prep complete
TROOTS-1  has taken off and is in transit
[BOOT] Adapted 2 ships into fleet_object
Seconds until arrival: 97.337349
Arrived and ready


NavigateShip200ResponseData(nav=ShipNav(system_symbol='X1-Q51', waypoint_symbol='X1-Q51-A3', route=ShipNavRoute(destination=ShipNavRouteWaypoint(symbol='X1-Q51-A3', type=<WaypointType.MOON: 'MOON'>, system_symbol='X1-Q51', x=3, y=23), origin=ShipNavRouteWaypoint(symbol='X1-Q51-K86', type=<WaypointType.PLANET: 'PLANET'>, system_symbol='X1-Q51', x=79, y=-67), departure_time=datetime.datetime(2025, 9, 14, 1, 3, 34, 818000, tzinfo=TzInfo(UTC)), arrival=datetime.datetime(2025, 9, 14, 1, 5, 12, 818000, tzinfo=TzInfo(UTC))), status=<ShipNavStatus.IN_TRANSIT: 'IN_TRANSIT'>, flight_mode=<ShipNavFlightMode.CRUISE: 'CRUISE'>), fuel=ShipFuel(current=282, capacity=400, consumed=ShipFuelConsumed(amount=118, timestamp=datetime.datetime(2025, 9, 14, 1, 3, 34, 825000, tzinfo=TzInfo(UTC)))), events=[])

In [28]:
from market_runtime import capture_market_for_waypoint
from db.auto_repo_sqlite import snapshot_many, TableSpec, upsert_many
from domain.market_rows import MarketGoodRow, MarketTransactionRow
import sqlite3
from services.arbitrage_repo import top_arbitrage

# get highest arbitrage data

ta = top_arbitrage()

#print(ta.iloc[0,0])
arbi_trade_symbol = ta.iloc[0,0]
arbi_buy_wp = ta.iloc[0,1]
arbi_buy_price = ta.iloc[0,2]
arbi_sell_wp = ta.iloc[0,3]
arbi_sell_price = ta.iloc[0,4]

print("This is trade symbol: ", arbi_trade_symbol)
print("This is the buy waypoint: ", arbi_buy_wp)
print("This is the buy price: ", arbi_buy_price)
print("This is the sell waypoint: ", arbi_sell_wp)
print("This is the sell price: ", arbi_sell_price)
print(ta)
#print(ta.at)


This is trade symbol:  MICROPROCESSORS
This is the buy waypoint:  X1-Q51-A3
This is the buy price:  2337
This is the sell waypoint:  X1-Q51-D42
This is the sell price:  3927
       trade_symbol buy_waypoint  buy_price sell_waypoint  sell_price  delta  \
0   MICROPROCESSORS    X1-Q51-A3       2337    X1-Q51-D42        3927   1590   
1              GOLD    X1-Q51-B7        216    X1-Q51-H54         363    147   
2              FUEL   X1-Q51-G50         45    X1-Q51-D41          68     23   
3  SILICON_CRYSTALS   X1-Q51-H53         37    X1-Q51-F49          47     10   
4   LIQUID_NITROGEN   X1-Q51-C39         32    X1-Q51-G50          40      8   
5   LIQUID_HYDROGEN   X1-Q51-C39         27    X1-Q51-G50          32      5   
6       QUARTZ_SAND   X1-Q51-H53         22    X1-Q51-F49          26      4   

                    buy_observed_at                  sell_observed_at  \
0  2025-09-13T04:02:38.281250+00:00  2025-09-13T11:10:44.156051+00:00   
1  2025-09-13T10:58:10.332794+00:00  20

In [ ]:

# get highest buy and sell market waypoints and good to be bought and sold

buy_wp = 
sell_wp = 
arbi_tradegood = 

# navigate to buy market


In [2]:

async def market_to_db(waypoint: str) -> None:
    # Fetch first (network I/O), then write to DB (short-lived connection)
    rows = await capture_market_for_waypoint(systems_api, waypoint)
    with sqlite3.connect("spacetraders.db") as conn:
        snapshot_many(conn, "market_goods", MarketGoodRow, rows["goods"])  # append-only history
        if rows["transactions"]:
            upsert_many(conn, TableSpec(table="market_transactions", pk="id"), rows["transactions"])



In [ ]:

traits = state.traits.by_wp
print(f"[BOOT] fleet={len(state.fleet.by_symbol)} ships, waypoints={len(state.waypoints.by_symbol)} (system), traits={sum(len(v) for v in state.traits.by_wp.values())}")
async def get_closest_wp_by_trait(trait: str):

    state = await init_world_state(fleet_api, agents_api, systems_api)
    traits = state.traits.by_wp
    data = []
    for wp_symbol, rows_list in traits.items():
        for row in rows_list:
            if row.trait_symbol == "MARKETPLACE":
                x = row.x or 0
                y = row.y or 0
                data.append({"waypoint": wp_symbol, "x": x, "y": y, "distance": math.hypot(x, y)})
    data = pd.DataFrame(data).sort_values("distance", ascending=True).head(50).reset_index(drop=True)
    #print (data.to_string(index=False))
    return data

markets_df = await get_closest_wp_by_trait("MARKETPLACE")
#print(markets_df)
print(markets_df.to_string(index=False))

"""
nav_resp = await api_navigate_ship(fleet_api, "TROOTS-1", "X1-Q51-H53")
print("This is nav_resp: ", nav_resp)
print("Sleeping for 4 seconds")
await asyncio.sleep(4)
nav_resp2 = await api_get_ship_nav(fleet_api, "TROOTS-1")
print("This is the current ship status: ", nav_resp2.status)
"""
#await api_navigate_ship(fleet_api, "TROOTS-1", "X1-Q51-H52")
"""
market_data = await capture_market_for_waypoint(systems_api,"X1-Q51-H53")
print(market_data)
"""


from market_runtime import capture_market_for_waypoint
from db.auto_repo_sqlite import snapshot_many, TableSpec, upsert_many
from domain.market_rows import MarketGoodRow, MarketTransactionRow
import sqlite3

async def market_to_db(waypoint: str) -> None:
    # Fetch first (network I/O), then write to DB (short-lived connection)
    rows = await capture_market_for_waypoint(systems_api, waypoint)
    with sqlite3.connect("spacetraders.db") as conn:
        snapshot_many(conn, "market_goods", MarketGoodRow, rows["goods"])  # append-only history
        if rows["transactions"]:
            upsert_many(conn, TableSpec(table="market_transactions", pk="id"), rows["transactions"])


# --- main patrol loop ---
async def patrol_markets(ship_symbol: str, markets_df) -> None:
    # dedupe and coerce to plain list of strings
    waypoints: List[str] = list(dict.fromkeys(markets_df["waypoint"].astype(str).tolist()))
    print(waypoints)
    if not waypoints:
        print("[WARN] No waypoints to patrol.")
        return

    print(f"[PATROL] {ship_symbol} looping through {len(waypoints)} markets.")
    idx = 0
    while True:
        wp = waypoints[idx]
        try:
            print(f"[PATROL] -> Navigating to {wp} (#{idx+1}/{len(waypoints)})")
            nav_resp = await api_navigate_ship(fleet_api, ship_symbol, wp)
            print(f"[MARKET] Capturing {wp} …")
            await market_to_db(wp)
            await asyncio.sleep(1.0)  # small dwell

        except asyncio.CancelledError:
            raise
        except Exception as e:
            print(f"[ERR] Patrol step at {wp} failed: {e!r}")
            await asyncio.sleep(3.0)  # brief backoff

        # round-robin
        idx = (idx + 1) % len(waypoints)


[BOOT] fleet=2 ships, waypoints=91 (system), traits=267
    waypoint    x   y   distance
  X1-SV25-A1  -24   8  25.298221
  X1-SV25-A2  -24   8  25.298221
  X1-SV25-A4  -24   8  25.298221
  X1-SV25-A3  -24   8  25.298221
X1-SV25-CX5E    8 -27  28.160256
 X1-SV25-H53   15 -43  45.541190
 X1-SV25-H54   15 -43  45.541190
 X1-SV25-H52   15 -43  45.541190
 X1-SV25-H55   15 -43  45.541190
 X1-SV25-E44   14  52  53.851648
 X1-SV25-E45   14  52  53.851648
 X1-SV25-G51   63  19  65.802736
 X1-SV25-F49  -70 -27  75.026662
 X1-SV25-F47  -70 -27  75.026662
 X1-SV25-F46  -70 -27  75.026662
 X1-SV25-D42  -16 -83  84.528102
 X1-SV25-D43  -16 -83  84.528102
 X1-SV25-K89  -29 102 106.042444
 X1-SV25-K90  -29 102 106.042444
 X1-SV25-K87  -29 102 106.042444
 X1-SV25-C41  114  11 114.529472
 X1-SV25-C40  156  15 156.719495
  X1-SV25-B6 -173  74 188.162164
 X1-SV25-I57  -63 221 229.804265
  X1-SV25-B7 -343  17 343.421024
 X1-SV25-I56 -125 434 451.642558
 X1-SV25-J58 -166 576 599.443075
 X1-SV25-J59 -199 69

CancelledError: 

In [ ]:

async def main():
    # You already computed `markets_df` above
    patroller = asyncio.create_task(patrol_markets("KIJINIBIBI-1", markets_df))
    await asyncio.gather(patroller)

await main()


In [10]:
import pprint
from services.arbitrage_repo  import best_arb_at_waypoint

nav_info = await api_get_ship_nav(fleet_api, "TROOTS-1")
cur_wp = nav_info.waypoint_symbol
print("Current waypoint is: ", cur_wp)

arb_cur_wp = best_arb_at_waypoint(cur_wp)
print("Arbitrage data: ", arb_cur_wp)

Current waypoint is:  X1-Q51-K86
Arbitrage data:  None


In [ ]:


from services.trade_runner import run_one_highest_delta_trade

async def trader_loop(ship_symbol: str):
    while True:
        try:
            summary = await run_one_highest_delta_trade(fleet_api, systems_api, ship_symbol, min_delta=2)
            if summary:
                print(summary)
            await asyncio.sleep(10)  # pacing; adjust as you like
        except Exception as e:
            print(f"[TRADE] error: {e!r}")
            await asyncio.sleep(5)

# Run both market patrol (scanning) & trading in parallel
async def main():
    # You already computed `markets_df` above
    trader = asyncio.create_task(trader_loop("TROOTS-1"))
    patroller = asyncio.create_task(patrol_markets("TROOTS-2", markets_df))
    await asyncio.gather(trader, patroller)

#await main()